# Step 5D.2d — UNSW-F0 tree-level forensic (перші 33 boosting-ітерації)

Мета: НЕ відтворити історичне `test_ap=0.973652` (це вже визнано наразі недосяжним у
поточному середовищі -- Step 5D.2c). Натомість -- покрокова реконструкція перших 33
boosting-ітерацій УЖЕ ВІДОМОГО, повторно навченого frozen UNSW-F0 protocol, щоб
локалізувати: (а) чи різні `default`/`deterministic` конфігурації розходяться на рівні
LightGBM `Dataset` (bin construction) ще ДО побудови жодного дерева, і (б) якщо ні -- на
якому саме дереві (1..33) вони вперше розходяться структурно.

Це НЕ новий експеримент і НЕ розморожування Step 5D.2. Для UNSW-F0 усе лишається
незмінним: той самий dedup, той самий temporal split, ті самі 41 ознака, ті самі
categorical-ознаки, `num_leaves=15`, `learning_rate=0.05`, `min_data_in_leaf=50`, `seed=42`.
Єдина зміна саме для цього forensic-кроку: `num_boost_round=33` БЕЗ early stopping (замість
`num_boost_round=500` + `early_stopping_rounds=30`) -- щоб отримати повну, невідфільтровану
trajectory для всіх 33 дерев, а не лише до точки зупинки.

Оригінальна (Step 5D.2) модель/дерева/trajectory тут так само НЕ відновлені -- вони ніколи
не зберігались (Step 5D.2c, розділ 7). Це порівняння `default` vs `deterministic` виконання
ПОТОЧНОГО середовища -- максимальна forensic-роздільність, доступна без історичного
артефакту.

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path.home() / "notebooks" / "2 article" / "release" / "results"
ENRICHED_DIR = Path("/srv/ids_research/data/enriched")
AUDIT_OUT_DIR = RESULTS_DIR / "step5d2_audit" / "unsw_f0_tree_forensic"
AUDIT_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("RESULTS_DIR:", RESULTS_DIR, "exists:", RESULTS_DIR.exists())
print("ENRICHED_DIR:", ENRICHED_DIR, "exists:", ENRICHED_DIR.exists())
print("AUDIT_OUT_DIR:", AUDIT_OUT_DIR)


RESULTS_DIR: /home/ivan_r_opirskyy/notebooks/2 article/release/results exists: True
ENRICHED_DIR: /srv/ids_research/data/enriched exists: True
AUDIT_OUT_DIR: /home/ivan_r_opirskyy/notebooks/2 article/release/results/step5d2_audit/unsw_f0_tree_forensic


## Step A -- Input fingerprint (UNSW-F0 only, frozen код, свіже завантаження)

Той самий код, що й у Step 5D.2b/5D.2c, обмежений лише UNSW-F0.

In [2]:
import re

FROZEN_BOUNDARIES_UNSW = dict(
    file="unsw_nb15_with_cve_context_dedup.parquet", time_col="Stime",
    label_col="attack_cat", benign_value="Normal",
    boundary_1=pd.Timestamp("2015-02-18 05:36:51"),
    boundary_2=pd.Timestamp("2015-02-18 08:58:36"),
)
FROZEN_SPLIT_SIZES_UNSW = dict(n_test=308_935, n_pos_test=25_354)

raw_unsw = pd.read_parquet(ENRICHED_DIR / FROZEN_BOUNDARIES_UNSW["file"])
print(f"UNSW: свіжо завантажено {len(raw_unsw):,} рядків, {len(raw_unsw.columns)} колонок")

IP_PATTERN = re.compile(r"^(srcip|src_ip|source_ip|dstip|dst_ip|destination_ip|src_addr|dst_addr|srcaddr|dstaddr)$", re.IGNORECASE)
PORT_PATTERN = re.compile(r"^(sport|src_port|source_port|dsport|dst_port|destination_port)$", re.IGNORECASE)
PROTO_PATTERN = re.compile(r"^(proto|protocol)$", re.IGNORECASE)
FLOWID_PATTERN = re.compile(r"(flow.?id|^id$)", re.IGNORECASE)
PROVENANCE_PATTERN = re.compile(r"^(__source_file|__.*)$", re.IGNORECASE)
TIME_COLS_UNSW = {"Stime", "Ltime"}
LABEL_COL_UNSW = "attack_cat"

def classify_column_unsw(col, df):
    if col.startswith("cve_ctx_"):
        return "CTI (cve_ctx_*)"
    if col == LABEL_COL_UNSW:
        return "target"
    if col in TIME_COLS_UNSW:
        return "time"
    if IP_PATTERN.match(col):
        return "network identifier (IP)"
    if PORT_PATTERN.match(col):
        return "network categorical feature (port)"
    if FLOWID_PATTERN.search(col):
        return "identifier (flow ID)"
    if PROVENANCE_PATTERN.match(col):
        return "provenance metadata"
    if col.lower() == "label":
        return "target-derived (redundant)"
    if PROTO_PATTERN.match(col):
        return "predictor (categorical)"
    if col.lower() == "service" and "cve_ctx_service" in df.columns:
        return "predictor (overlap risk)"
    return "predictor"

INCLUDE_ROLES = {"predictor", "predictor (categorical)", "predictor (overlap risk)"}
roles_unsw = {col: classify_column_unsw(col, raw_unsw) for col in raw_unsw.columns}
F0_UNSW = [c for c, r in roles_unsw.items() if r in INCLUDE_ROLES]
assert len(F0_UNSW) == 41, f"UNSW F0 = {len(F0_UNSW)}, очікувалось 41"
print(f"✅ UNSW F0 = {len(F0_UNSW)} predictors (frozen)")


UNSW: свіжо завантажено 2,059,418 рядків, 62 колонок
✅ UNSW F0 = 41 predictors (frozen)


In [3]:
df_unsw = raw_unsw.copy()
df_unsw["_is_attack"] = (df_unsw[FROZEN_BOUNDARIES_UNSW["label_col"]] != FROZEN_BOUNDARIES_UNSW["benign_value"]).astype(int)

if "ct_ftp_cmd" in df_unsw.columns:
    df_unsw["ct_ftp_cmd"] = pd.to_numeric(df_unsw["ct_ftp_cmd"].replace(" ", np.nan), errors="coerce")

t_unsw = pd.to_datetime(df_unsw["Stime"], unit="s", errors="raise") \
    if pd.api.types.is_integer_dtype(df_unsw["Stime"]) else pd.to_datetime(df_unsw["Stime"])

train_mask = t_unsw < FROZEN_BOUNDARIES_UNSW["boundary_1"]
val_mask = (t_unsw >= FROZEN_BOUNDARIES_UNSW["boundary_1"]) & (t_unsw < FROZEN_BOUNDARIES_UNSW["boundary_2"])
test_mask = t_unsw >= FROZEN_BOUNDARIES_UNSW["boundary_2"]

train_df = df_unsw[train_mask].copy()
val_df = df_unsw[val_mask].copy()
test_df = df_unsw[test_mask].copy()

n_test, n_pos_test = len(test_df), int(test_df["_is_attack"].sum())
assert n_test == FROZEN_SPLIT_SIZES_UNSW["n_test"]
assert n_pos_test == FROZEN_SPLIT_SIZES_UNSW["n_pos_test"]
print(f"✅ split точно збігається з frozen Step 5C.2 (n_test={n_test:,}, n_pos_test={n_pos_test:,})")

categorical_candidates = [c for c in F0_UNSW
                           if not pd.api.types.is_numeric_dtype(df_unsw[c]) or df_unsw[c].dtype == bool]
assert "ct_ftp_cmd" not in categorical_candidates
train_categories = {}
for col in categorical_candidates:
    cats = pd.Categorical(train_df[col]).categories
    train_categories[col] = cats
    train_df[col] = pd.Categorical(train_df[col], categories=cats)
    val_df[col] = pd.Categorical(val_df[col], categories=cats)
    test_df[col] = pd.Categorical(test_df[col], categories=cats)

X_train, y_train = train_df[F0_UNSW], train_df["_is_attack"]
X_val, y_val = val_df[F0_UNSW], val_df["_is_attack"]
X_test, y_test = test_df[F0_UNSW], test_df["_is_attack"]

n_pos_train = int(y_train.sum())
n_neg_train = len(y_train) - n_pos_train
SCALE_POS_WEIGHT = n_neg_train / n_pos_train

print(f"\nКатегоріальні ознаки F0: {categorical_candidates}")
print(f"train: n={len(train_df):,}  positive={n_pos_train:,}  negative={n_neg_train:,}")
print(f"scale_pos_weight = {SCALE_POS_WEIGHT:.4f}")
assert abs(SCALE_POS_WEIGHT - 28.2135) < 0.01, (
    f"scale_pos_weight={SCALE_POS_WEIGHT:.4f} суттєво відрізняється від очікуваного 28.2135 -- "
    f"це саме по собі forensic-сигнал (інший train split чи інша база позитивів/негативів)."
)
print("✅ scale_pos_weight збігається з очікуваним 28.2135 (у межах округлення)")


✅ split точно збігається з frozen Step 5C.2 (n_test=308,935, n_pos_test=25,354)

Категоріальні ознаки F0: ['proto', 'state', 'service']
train: n=1,441,568  positive=49,346  negative=1,392,222
scale_pos_weight = 28.2135
✅ scale_pos_weight збігається з очікуваним 28.2135 (у межах округлення)


## Step B -- LightGBM `Dataset` fingerprint (num_data, num_feature, Total Bins) для ОБОХ конфігурацій

`Total Bins` захоплюється через `lgb.register_logger()` (перевірено окремо, що це надійно
парсить рядок `Total Bins N` з C++-логу LightGBM під час `lgb.train`). Це формалізує вже
знайдений сигнал (Step 5D.2c): 5715 (`default`) vs 5677 (`deterministic`) -- тепер з явним
джерелом даних і можливістю відтворити цю перевірку в майбутньому.

In [4]:
import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score

RANDOM_STATE = 42
LGB_PARAMS = dict(
    objective="binary", metric="None", seed=RANDOM_STATE,
    num_leaves=15, learning_rate=0.05, min_data_in_leaf=50,
    scale_pos_weight=SCALE_POS_WEIGHT,
)
DETERMINISTIC_PARAM_OVERRIDES = dict(
    bagging_seed=RANDOM_STATE, feature_fraction_seed=RANDOM_STATE, data_random_seed=RANDOM_STATE,
    deterministic=True, force_row_wise=True,
)
NUM_BOOST_ROUND_FORENSIC = 33  # той самий historical best_iteration -- БЕЗ early stopping

def unweighted_average_precision(preds, train_data):
    y_true = train_data.get_label()
    return "ap", average_precision_score(y_true, preds), True


class ListLogger:
    def __init__(self):
        self.lines = []
    def info(self, msg):
        self.lines.append(str(msg))
    def warning(self, msg):
        self.lines.append(str(msg))


def capture_total_bins(params, cat_cols):
    logger = ListLogger()
    lgb.register_logger(logger)
    ds = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols if cat_cols else "auto")
    tmp_model = lgb.train(params, ds, num_boost_round=1)
    log_text = "\n".join(logger.lines)
    m = re.search(r"Total Bins (\d+)", log_text)
    total_bins = int(m.group(1)) if m else None
    return total_bins, ds.num_data(), ds.num_feature()


cat_cols_f0 = [c for c in F0_UNSW if X_train[c].dtype.name == "category"]

dataset_fingerprints = {}
for config_name, overrides in [("default", None), ("deterministic", DETERMINISTIC_PARAM_OVERRIDES)]:
    params = dict(LGB_PARAMS)
    if overrides:
        params.update(overrides)
    total_bins, num_data, num_feature = capture_total_bins(params, cat_cols_f0)
    dataset_fingerprints[config_name] = dict(
        total_bins=total_bins, num_data=num_data, num_feature=num_feature,
        categorical_features=cat_cols_f0,
    )
    print(f"[{config_name}]  Total Bins={total_bins}  num_data={num_data}  "
          f"num_feature={num_feature}  categorical={cat_cols_f0}")

if dataset_fingerprints["default"]["total_bins"] != dataset_fingerprints["deterministic"]["total_bins"]:
    print(
        "\n❗ ПІДТВЕРДЖЕНО: Total Bins РІЗНИТЬСЯ між default і deterministic конфігураціями "
        "при ІДЕНТИЧНИХ вхідних даних, ознаках і seed=42. Розбіжність виникає ДО побудови "
        "жодного дерева -- на етапі bin construction самого LightGBM Dataset."
    )
else:
    print("\n✅ Total Bins однаковий між конфігураціями -- джерело розбіжності НЕ в bin construction.")


[default]  Total Bins=5715  num_data=1441568  num_feature=41  categorical=['proto', 'state', 'service']
[deterministic]  Total Bins=5677  num_data=1441568  num_feature=41  categorical=['proto', 'state', 'service']

❗ ПІДТВЕРДЖЕНО: Total Bins РІЗНИТЬСЯ між default і deterministic конфігураціями при ІДЕНТИЧНИХ вхідних даних, ознаках і seed=42. Розбіжність виникає ДО побудови жодного дерева -- на етапі bin construction самого LightGBM Dataset.


## Step C -- Тренування на 33 дерева (без early stopping), метод 1: `num_iteration=i` з ОДНОГО model object

Для обох конфігурацій (`default`, `deterministic`) тренуємо ОДНУ модель на 33 дерева, а
потім отримуємо trajectory через `model.predict(X_val, num_iteration=i)` для `i=1..33` --
це дає validation AP/ROC-AUC та статистику прогнозів (mean/std/min/max/n_unique) на кожній
ітерації з ОДНОГО навченого об'єкта моделі.

In [5]:
def train_fixed_33(param_overrides=None):
    params = dict(LGB_PARAMS)
    if param_overrides:
        params.update(param_overrides)
    lgb_train = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols_f0 if cat_cols_f0 else "auto")
    lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train, categorical_feature=cat_cols_f0 if cat_cols_f0 else "auto")
    model = lgb.train(
        params, lgb_train, num_boost_round=NUM_BOOST_ROUND_FORENSIC,
        valid_sets=[lgb_val], valid_names=["val"],
        feval=unweighted_average_precision,
        callbacks=[lgb.log_evaluation(period=0)],
    )
    return model


def trajectory_method1(model):
    rows = []
    for i in range(1, NUM_BOOST_ROUND_FORENSIC + 1):
        preds_val = model.predict(X_val, num_iteration=i)
        preds_test = model.predict(X_test, num_iteration=i)
        rows.append(dict(
            iteration=i,
            val_ap=average_precision_score(y_val, preds_val),
            val_roc_auc=roc_auc_score(y_val, preds_val),
            test_ap=average_precision_score(y_test, preds_test),
            test_roc_auc=roc_auc_score(y_test, preds_test),
            val_pred_mean=float(np.mean(preds_val)), val_pred_std=float(np.std(preds_val)),
            val_pred_min=float(np.min(preds_val)), val_pred_max=float(np.max(preds_val)),
            val_pred_n_unique=int(np.unique(np.round(preds_val, 10)).size),
        ))
    return pd.DataFrame(rows)


models_33 = {}
trajectories_m1 = {}
for config_name, overrides in [("default", None), ("deterministic", DETERMINISTIC_PARAM_OVERRIDES)]:
    print(f"\nТренування UNSW-F0 [{config_name}], num_boost_round=33, без early stopping...")
    model = train_fixed_33(overrides)
    models_33[config_name] = model
    traj = trajectory_method1(model)
    trajectories_m1[config_name] = traj
    print(traj[["iteration", "val_ap", "test_ap", "val_pred_n_unique"]].to_string(index=False))



Тренування UNSW-F0 [default], num_boost_round=33, без early stopping...
 iteration   val_ap  test_ap  val_pred_n_unique
         1 0.922971 0.936437                 15
         2 0.923655 0.936613                 44
         3 0.923655 0.936603                 58
         4 0.939097 0.950168                 89
         5 0.943221 0.953614                124
         6 0.944495 0.955212                145
         7 0.945902 0.956016                197
         8 0.946507 0.956347                214
         9 0.946284 0.956178                214
        10 0.949285 0.959247                306
        11 0.949588 0.959409                378
        12 0.949938 0.959752                389
        13 0.951755 0.961535                452
        14 0.958874 0.967769                787
        15 0.960536 0.968978                885
        16 0.960499 0.969058                909
        17 0.961207 0.969884               1156
        18 0.961187 0.969790               1169
        19 0.96

## Step C2 -- Контрольний тест: послідовне тренування дерево-за-деревом (`init_model`) для `default`

Якщо метод 1 (один прогін на 33 дерева, `predict(num_iteration=i)`) і метод 2 (33 окремих
виклики `lgb.train(num_boost_round=1, init_model=...)`) дають ІДЕНТИЧНУ trajectory -- це
виключає ще один клас можливих артефактів (щось специфічне для послідовного `init_model`
ланцюга). Виконується лише для `default` (щоб контролювати час виконання; `deterministic`
конфігурація вже незалежно підтверджена стабільною в Step 5D.2b, розділ 7d).

In [6]:
def trajectory_method2_sequential(param_overrides=None):
    params = dict(LGB_PARAMS)
    if param_overrides:
        params.update(param_overrides)
    # free_raw_data=False -- обов'язково для послідовного init_model-ланцюга: LightGBM за
    # замовчуванням звільняє сирі дані Dataset після train(), а наступний виклик з
    # init_model потребує їх повторно для перерахунку predictor init scores.
    lgb_train = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols_f0 if cat_cols_f0 else "auto",
                             free_raw_data=False)
    lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train, categorical_feature=cat_cols_f0 if cat_cols_f0 else "auto",
                           free_raw_data=False)

    rows = []
    model = None
    for i in range(1, NUM_BOOST_ROUND_FORENSIC + 1):
        model = lgb.train(
            params, lgb_train, num_boost_round=1, init_model=model,
            valid_sets=[lgb_val], valid_names=["val"],
            feval=unweighted_average_precision,
            callbacks=[lgb.log_evaluation(period=0)],
        )
        preds_val = model.predict(X_val, num_iteration=model.current_iteration())
        preds_test = model.predict(X_test, num_iteration=model.current_iteration())
        rows.append(dict(
            iteration=i,
            val_ap=average_precision_score(y_val, preds_val),
            val_roc_auc=roc_auc_score(y_val, preds_val),
            test_ap=average_precision_score(y_test, preds_test),
            test_roc_auc=roc_auc_score(y_test, preds_test),
        ))
    return pd.DataFrame(rows), model


print("Послідовне тренування дерево-за-деревом (default), 33 виклики lgb.train(num_boost_round=1)...")
traj_m2_default, model_m2_default = trajectory_method2_sequential(None)
print(traj_m2_default[["iteration", "val_ap", "test_ap"]].to_string(index=False))

cmp_df = trajectories_m1["default"][["iteration", "val_ap", "test_ap"]].merge(
    traj_m2_default[["iteration", "val_ap", "test_ap"]], on="iteration", suffixes=("_m1", "_m2")
)
cmp_df["val_ap_diff"] = (cmp_df["val_ap_m1"] - cmp_df["val_ap_m2"]).abs()
cmp_df["test_ap_diff"] = (cmp_df["test_ap_m1"] - cmp_df["test_ap_m2"]).abs()
max_diff = cmp_df[["val_ap_diff", "test_ap_diff"]].max().max()
print(f"\nМаксимальна розбіжність (val_ap/test_ap) між методом 1 (один прогін, num_iteration=i) "
      f"і методом 2 (послідовний init_model) по всіх 33 ітераціях: {max_diff:.2e}")
if max_diff < 1e-9:
    print("✅ Метод 1 і метод 2 дають БІТОВО ІДЕНТИЧНУ trajectory -- виключає артефакти, специфічні "
          "для init_model-ланцюга чи для num_iteration-зрізу одного model object.")
else:
    print("❌ Методи РОЗХОДЯТЬСЯ -- сам спосіб реконструкції trajectory впливає на результат, "
          "це окреме джерело недетермінізму, яке треба дослідити перед подальшими висновками.")


Послідовне тренування дерево-за-деревом (default), 33 виклики lgb.train(num_boost_round=1)...
 iteration   val_ap  test_ap
         1 0.922971 0.936437
         2 0.923655 0.936613
         3 0.923655 0.936603
         4 0.939097 0.950168
         5 0.943222 0.953614
         6 0.944497 0.955213
         7 0.945902 0.956016
         8 0.946544 0.956376
         9 0.946284 0.956178
        10 0.949285 0.959248
        11 0.949590 0.959403
        12 0.949945 0.959753
        13 0.951621 0.961431
        14 0.958819 0.967729
        15 0.960478 0.968912
        16 0.960470 0.969031
        17 0.961191 0.969868
        18 0.961205 0.969789
        19 0.961701 0.970230
        20 0.961804 0.970297
        21 0.961896 0.970277
        22 0.963794 0.972031
        23 0.963427 0.972407
        24 0.963707 0.972697
        25 0.963853 0.972882
        26 0.964055 0.973193
        27 0.964170 0.973318
        28 0.964199 0.973258
        29 0.964227 0.973328
        30 0.964224 0.973295
       

## Step D -- Tree-level структурний fingerprint (дерева 1-33, `default` vs `deterministic`)

Кожне дерево з `model.dump_model()['tree_info']` хешується цілком (SHA-256 від канонічного
JSON `tree_structure`) -- це найповніший можливий fingerprint дерева без порівняння з
історичним артефактом (якого не існує). Додатково виводиться human-readable підсумок:
кореневий split feature, поріг, кількість листків -- щоб знайти ПЕРШЕ дерево, на якому
`default` і `deterministic` розходяться структурно.

In [7]:
import hashlib

def tree_fingerprints(model):
    dumped = model.dump_model()
    feature_names = dumped["feature_names"]
    rows = []
    for tree in dumped["tree_info"]:
        struct = tree["tree_structure"]
        tree_hash = hashlib.sha256(json.dumps(struct, sort_keys=True).encode()).hexdigest()
        root_feature_idx = struct.get("split_feature")
        root_feature_name = feature_names[root_feature_idx] if root_feature_idx is not None else None
        rows.append(dict(
            tree_index=tree["tree_index"],
            num_leaves=tree["num_leaves"],
            root_split_feature=root_feature_name,
            root_threshold=struct.get("threshold"),
            root_decision_type=struct.get("decision_type"),
            tree_hash=tree_hash,
        ))
    return pd.DataFrame(rows)


tree_fp = {config_name: tree_fingerprints(models_33[config_name]) for config_name in models_33}

compare_trees = tree_fp["default"].merge(
    tree_fp["deterministic"], on="tree_index", suffixes=("_default", "_deterministic")
)
compare_trees["structurally_identical"] = compare_trees["tree_hash_default"] == compare_trees["tree_hash_deterministic"]

print(compare_trees[[
    "tree_index", "root_split_feature_default", "root_split_feature_deterministic",
    "num_leaves_default", "num_leaves_deterministic", "structurally_identical",
]].to_string(index=False))

first_divergence = compare_trees.loc[~compare_trees["structurally_identical"], "tree_index"]
if first_divergence.empty:
    print("\n✅ Усі 33 дерева СТРУКТУРНО ІДЕНТИЧНІ між default і deterministic -- розбіжність "
          "best_iteration (277 vs 267 у Step 5D.2c) виникає ПІЗНІШЕ, за межами перших 33 дерев.")
else:
    first_idx = int(first_divergence.iloc[0])
    print(f"\n❗ ПЕРШЕ структурно відмінне дерево: tree_index={first_idx} "
          f"(0-indexed; це {first_idx + 1}-е дерево). До цього дерева default і deterministic "
          f"будують СТРУКТУРНО ІДЕНТИЧНІ дерева, попри різний Total Bins (Step B).")


 tree_index root_split_feature_default root_split_feature_deterministic  num_leaves_default  num_leaves_deterministic  structurally_identical
          0                       sttl                             sttl                  15                        15                   False
          1                       sttl                             sttl                  15                        15                   False
          2                       sttl                             sttl                  15                        15                   False
          3                       sttl                             sttl                  15                        15                   False
          4                       sttl                             sttl                  15                        15                   False
          5                       sttl                             sttl                  15                        15                   False
      

## Явна межа цього forensic-кроку

Усе вище -- порівняння `default` vs `deterministic` ВИКОНАННЯ В ПОТОЧНОМУ середовищі.
Історичні (Step 5D.2, оригінальний прогін) дерева, `Dataset` fingerprint чи trajectory
НЕ відновлені й не можуть бути відновлені -- вони ніколи не зберігались. Це та сама
задокументована межа, що й у Step 5D.2c, повторена тут явно, щоб жоден висновок нижче не
читався як "ми з'ясували, чим саме оригінал відрізнявся від поточного".

In [8]:
print(
    "❌ Оригінальна (Step 5D.2) LightGBM Dataset fingerprint, tree structures та trajectory: "
    "НЕ ВІДНОВЛЕНІ. Усі порівняння в цьому notebook -- це default vs deterministic ВСЕРЕДИНІ "
    "поточного середовища, а не проти історичного прогону."
)


❌ Оригінальна (Step 5D.2) LightGBM Dataset fingerprint, tree structures та trajectory: НЕ ВІДНОВЛЕНІ. Усі порівняння в цьому notebook -- це default vs deterministic ВСЕРЕДИНІ поточного середовища, а не проти історичного прогону.


## Збереження форензик-артефактів

In [9]:
for config_name, traj in trajectories_m1.items():
    traj.to_csv(AUDIT_OUT_DIR / f"unsw_f0_trajectory_method1_{config_name}.csv", index=False)
traj_m2_default.to_csv(AUDIT_OUT_DIR / "unsw_f0_trajectory_method2_sequential_default.csv", index=False)
for config_name, fp_df in tree_fp.items():
    fp_df.to_csv(AUDIT_OUT_DIR / f"unsw_f0_tree_fingerprints_{config_name}.csv", index=False)
compare_trees.to_csv(AUDIT_OUT_DIR / "unsw_f0_tree_fingerprints_comparison.csv", index=False)

summary = dict(
    scale_pos_weight=SCALE_POS_WEIGHT,
    dataset_fingerprints=dataset_fingerprints,
    method1_vs_method2_max_diff=float(max_diff),
    method1_vs_method2_identical=bool(max_diff < 1e-9),
    first_structural_tree_divergence=(
        int(first_divergence.iloc[0]) if not first_divergence.empty else None
    ),
    all_33_trees_structurally_identical=bool(first_divergence.empty),
    original_dataset_fingerprint_recovered=False,
    original_tree_structures_recovered=False,
)
with open(AUDIT_OUT_DIR / "unsw_f0_tree_forensic_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Збережено в {AUDIT_OUT_DIR}:")
for p in sorted(AUDIT_OUT_DIR.glob("*")):
    print(f"  {p.name}")


Збережено в /home/ivan_r_opirskyy/notebooks/2 article/release/results/step5d2_audit/unsw_f0_tree_forensic:
  unsw_f0_trajectory_method1_default.csv
  unsw_f0_trajectory_method1_deterministic.csv
  unsw_f0_trajectory_method2_sequential_default.csv
  unsw_f0_tree_fingerprints_comparison.csv
  unsw_f0_tree_fingerprints_default.csv
  unsw_f0_tree_fingerprints_deterministic.csv
  unsw_f0_tree_forensic_summary.json


## Підсумок цього forensic-кроку

Заповнюється фактичним виводом після запуску:

- **Step B**: чи `Total Bins` дійсно різний між `default`/`deterministic` (уже відомо з
  Step 5D.2c -- 5715 vs 5677) -- тут формалізовано з явним джерелом.
- **Step C2**: чи метод 1 (один прогін, `num_iteration=i`) і метод 2 (послідовний
  `init_model`) дають ідентичну trajectory -- якщо ні, це ОКРЕМЕ джерело недетермінізму,
  яке треба дослідити раніше за все інше.
- **Step D**: номер першого структурно відмінного дерева між `default` і `deterministic`
  -- якщо розбіжність в `Total Bins` (Step B) не виливається у структурну розбіжність
  перших 33 дерев, це означає, що bin-різниця стосується ознак, які не використовуються
  (або використовуються ідентично) в перших 33 splits, і "справжня" причина відмінного
  `best_iteration` (277 vs 267 vs історичні 33) лежить десь у пізніших дереваx або в
  накопичувальному ефекті, а не в перших 33 ітераціях як таких.

Незалежно від результату: історичний `best_iteration=33` і `test_ap=0.973652` лишаються
НЕ відтвореними -- цей notebook лише характеризує поведінку ПОТОЧНОГО frozen protocol, а не
відновлює оригінал.